<a href="https://colab.research.google.com/github/lexinejazly-asuncion/Sign-Language-Classification/blob/main/Sign_Language_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas
!pip install pillow
!pip install kaggle
!pip install scikit-learn
!pip install torchvision

You should consider upgrading via the '/Users/nelly/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/nelly/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/nelly/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/nelly/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/nelly/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import sys
!{sys.executable} -m pip install --upgrade pandas numexpr bottleneck
!{sys.executable} -m pip install "numpy<2"

You should consider upgrading via the '/Users/nelly/Sign-Language-Classification/venv/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/nelly/Sign-Language-Classification/venv/bin/python -m pip install --upgrade pip' command.


In [3]:
from pathlib import Path
import zipfile

dataset_folder = Path("dataset/SigNN Character Database")
zip_path = Path("dataset/asl-sign-language-pictures-minus-j-z.zip")

if not zip_path.exists():
    !kaggle datasets download -d signnteam/asl-sign-language-pictures-minus-j-z -p dataset
else:
    print("Zip file already downloaded.")

if not dataset_folder.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("dataset")
    print("Dataset extracted.")
else:
    print("Dataset already extracted.")

Zip file already downloaded.
Dataset already extracted.


## **EDA**

In [4]:
import pandas as pd
from pathlib import Path

In [5]:
data_path = Path("dataset/SigNN Character Database")

paths = [path.parts[-2:] for path in data_path.rglob("*.*")] #gives the class and image file name

sl_df = pd.DataFrame(data=paths, columns=["Class","Images"]) #create column names for dataframe
sl_df = sl_df.sort_values("Class",ascending=True)
sl_df.reset_index(drop=True, inplace=True)
sl_df.head()

,Class,Images
0,A,314.jpg
1,A,536.jpg
2,A,250.jpg
3,A,278.jpg
4,A,279.jpg


In [6]:
print(f"Total images in dataset: {len(sl_df)}")
print(f"Total classes: {sl_df['Class'].nunique()}")
print("\nTotal images per class:")
print(sl_df["Class"].value_counts().to_dict())

Total images in dataset: 8442
Total classes: 24

Total images per class:
{'B': 541, 'A': 539, 'E': 498, 'F': 420, 'C': 387, 'D': 379, 'O': 374, 'H': 364, 'I': 360, 'W': 347, 'L': 346, 'G': 345, 'V': 337, 'K': 319, 'Y': 318, 'S': 314, 'X': 310, 'T': 301, 'N': 293, 'R': 291, 'U': 286, 'M': 277, 'Q': 275, 'P': 221}


In [7]:
from PIL import Image
import os

#finds dimensions of each image
def get_img_dim(x):
  img_path = os.path.join(data_path, x["Class"], x["Images"])
  with Image.open(img_path) as img:
    return img.size

#creates two new columns in df: width and height
sl_df[["Width", "Height"]] = sl_df.apply(get_img_dim, axis=1, result_type="expand")
sl_df.head()

,Class,Images,Width,Height
0,A,314.jpg,640,480
1,A,536.jpg,1920,1920
2,A,250.jpg,640,480
3,A,278.jpg,640,480
4,A,279.jpg,640,480


In [8]:
#finds the different dimensions of each class
img_sizes = sl_df.groupby('Class').apply(
    lambda x: x[['Width', 'Height']].drop_duplicates().apply(
        lambda x: f"{int(x['Width'])}x{int(x['Height'])}", axis=1).tolist(), include_groups=False
)

print(f"Image sizes per class: {img_sizes}")


Image sizes per class: Class
A              [640x480, 1920x1920]
B              [1920x1920, 640x480]
C              [640x480, 1920x1920]
D              [640x480, 1920x1920]
E              [640x480, 1920x1920]
F              [640x480, 1920x1920]
G              [640x480, 1920x1920]
H              [640x480, 1920x1920]
I              [1920x1920, 640x480]
K              [640x480, 1920x1920]
L              [640x480, 1920x1920]
M              [640x480, 1920x1920]
N    [640x480, 1920x1920, 1280x720]
O              [640x480, 1920x1920]
P              [640x480, 1920x1920]
Q    [640x480, 1280x720, 1920x1920]
R              [1920x1920, 640x480]
S    [640x480, 1920x1920, 1280x720]
T              [640x480, 1920x1920]
U              [640x480, 1920x1920]
V    [640x480, 1920x1920, 1280x720]
W    [640x480, 1280x720, 1920x1920]
X              [640x480, 1920x1920]
Y              [640x480, 1920x1920]
dtype: object


## **Preprocessing**

In [9]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

In [10]:
train, test = train_test_split(sl_df, test_size=0.2, stratify=sl_df["Class"], random_state=42)

train_set = train.groupby("Class").sample(n=150, random_state=42).reset_index(drop=True)

In [11]:
class ASLDataset(Dataset):
    def __init__(self, df, data_path, transform=None):
        self.df = df
        self.data_path = data_path
        self.transform = transform
        self.label_map = {label: i for i, label in enumerate(sorted(df['Class'].unique()))} #for label encoding
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.data_path, row["Class"], row["Images"])
        
        image = Image.open(img_path).convert("RGB")
        label = self.label_map[row["Class"]] #labels represented as ints
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

preprocess_pipeline = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [12]:
#create the dataset object using the train_set (the 150 samples per class)
train_dataset = ASLDataset(train_set, data_path, transform=preprocess_pipeline)

#apply preprocessing
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [13]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(labels[:10])

torch.Size([32, 3, 128, 128])
torch.Size([32])
tensor([ 6,  3, 20, 16, 11,  6,  4, 17, 11, 21])


## **Training**

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score

test_dataset = ASLDataset(test, data_path, transform=preprocess_pipeline)
test_dataset.label_map = train_dataset.label_map

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class ASLCNN(nn.Module):
    def __init__(self, num_classes=24):
        super(ASLCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ASLCNN(num_classes=24).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {running_loss / len(train_loader):.4f}")

Epoch 1/5, Loss: 2.5429
Epoch 2/5, Loss: 1.1174
Epoch 3/5, Loss: 0.6328
Epoch 4/5, Loss: 0.4274
Epoch 5/5, Loss: 0.3094


In [15]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")

Test Accuracy: 0.8555
Weighted F1 Score: 0.8560


In [ ]:
from torchvision import models, transforms
import torch
import torch.nn as nn
import torch.optim as optim

# ResNet preprocessing
resnet_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = ASLDataset(train_set, data_path, transform=resnet_preprocess)
test_dataset = ASLDataset(test, data_path, transform=resnet_preprocess)
test_dataset.label_map = train_dataset.label_map

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model = models.resnet18(pretrained=True)


for param in model.parameters():
    param.requires_grad = False

#had to unfreeze layer4 to get better performance, otherwise the model was underfitting was 
#getting around 80 %accuracy but after unfreezing layer4, accuracy jumped to around 97%
for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, 24)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 0.0001},
    {"params": model.fc.parameters(), "lr": 0.001}
])
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/5, Loss: 1.2027
Epoch 2/5, Loss: 0.1343
Epoch 3/5, Loss: 0.0352
Epoch 4/5, Loss: 0.0176
Epoch 5/5, Loss: 0.0112


In [19]:

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")

Test Accuracy: 0.9668
Weighted F1 Score: 0.9669
